# V4.3 Strict Baseline × GraphSAGE + GINE × Abs-Slope NE × V4.2 oldRW 消融實驗

這份 notebook 把 **V4.2 oldRW / Balanced Dirty Exposure RW** 放進目前的 **strict baseline framework**，同時跑兩個 backbone：

```text
1. GraphSAGE-Max / PhysicsSAGE-style
2. GINE-edge
```

原則：

```text
build_graph_material(...) 不動
build_ne_features(...) 僅修改 potential-work：先對狀態坡度取絕對值，再與其餘正向做功項相加
oldRW 只新增為 diagnostic / V4.2-compatible RW function
```

oldRW 設定：

```text
dirty walkers = 20620
clean walkers = 20620
total walkers = 41240
seeds only from train positives / train negatives
output = 1D scalar dirty exposure
```

四組 strict 消融：

| Mode | Node features | Edge features |
|---|---|---|
| `base` | `ne["node_native"]` | raw edge subset 40 |
| `base_ne` | `ne["node_features"]` | full edge 43 |
| `base_rw_old` | `ne["node_native"] + oldRW scalar` | raw edge subset 40 |
| `base_nerw_old` | `ne["node_features"] + oldRW scalar` | full edge 43 |

預期維度：

```text
base:          node=21, edge=40
base_ne:       node=34, edge=43
base_rw_old:   node=22, edge=40
base_nerw_old: node=35, edge=43
```

訓練口徑：

```text
Val PR-AUC 選 best checkpoint
Val best-F1 threshold 套到 Test
epochs = 750
eval_every = 10
seed = 42
```


> **本版本的唯一方法變更**  
> 原式允許 signed potential work 與 friction / kinetic work 互相抵銷；本版本改為
> `w_potential = amount * abs(grad_z)`，因此三個做功項皆為非負值，不再發生正負抵銷。
> `slope_log` 仍保留原本的有號高度差，因此下游模型仍可觀察交易方向。


## 0. 安裝環境

In [ ]:

import importlib.util
import subprocess
import sys

required_packages = {
    "torch_geometric": "torch-geometric",
    "kagglehub": "kagglehub",
    "sklearn": "scikit-learn",
}

missing = [pip_name for module_name, pip_name in required_packages.items()
           if importlib.util.find_spec(module_name) is None]

if missing:
    print("安裝缺少套件：", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("✅ 必要套件已存在。")


## 1. 從 Google Drive 優先取得交易資料

In [ ]:

import os
import glob
import pandas as pd

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except Exception as exc:
    print(f"非 Colab 或 Drive 無法掛載：{exc}")

DRIVE_DATA_DIR = "/content/drive/MyDrive/AML_Data/IBM_AML"
CSV_NAME = "HI-Small_Trans.csv"
NROWS = 5_000_000


def locate_transaction_csv(
    csv_name=CSV_NAME,
    drive_data_dir=DRIVE_DATA_DIR,
):
    """優先找 Google Drive；找不到才使用 KaggleHub；最後嘗試目前目錄。"""
    direct_path = os.path.join(drive_data_dir, csv_name)
    if os.path.exists(direct_path):
        print(f"✅ Google Drive 找到：{direct_path}")
        return direct_path

    if os.path.exists(drive_data_dir):
        candidates = glob.glob(
            os.path.join(drive_data_dir, "**", csv_name),
            recursive=True,
        )
        if candidates:
            print(f"✅ Google Drive 子資料夾找到：{candidates[0]}")
            return candidates[0]

    print("Drive 找不到資料，改用 KaggleHub。")
    try:
        import kagglehub
        dataset_dir = kagglehub.dataset_download(
            "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
        )
        kaggle_path = os.path.join(dataset_dir, csv_name)
        if os.path.exists(kaggle_path):
            print(f"✅ KaggleHub 找到：{kaggle_path}")
            return kaggle_path
    except Exception as exc:
        print(f"⚠️ KaggleHub 失敗：{exc}")

    if os.path.exists(csv_name):
        print(f"✅ 使用目前工作目錄：{csv_name}")
        return csv_name

    raise FileNotFoundError(
        f"找不到 {csv_name}。請放到 {drive_data_dir} 或目前工作目錄。"
    )


DATA_PATH = locate_transaction_csv()
print(f"讀取：{DATA_PATH}")
df_raw = pd.read_csv(DATA_PATH, nrows=NROWS)
df_raw.columns = df_raw.columns.str.strip()
print(f"✅ df_raw = {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(df_raw.columns.tolist())


## 2. 主函數一：胚料 `build_graph_material`

In [ ]:

from __future__ import annotations

import numpy as np
import pandas as pd
import torch


def build_graph_material(
    df_raw: pd.DataFrame,
    *,
    use_bank_account_id: bool = True,
    burn_in_days: int = 1,
    snapshot_ratio: float = 0.8,
    split_mode: str = "temporal_node",
    split_ratios: tuple[float, float, float] = (0.6, 0.2, 0.2),
    split_seed: int = 42,
) -> dict:
    """
    原始交易表 → 統一圖胚料。

    此函數集中負責：
    1. 欄位正規化與 bank::account node id。
    2. 穩定交易 ID、時間排序、burn-in 與 snapshot。
    3. 唯一 node_idx / edge_idx / edge_index。
    4. node label 與 train/val/test mask。
    5. RW 唯一允許使用的 train-known dirty mask。

    它不計算 NE，也不執行 RW。
    """
    required = {
        "Timestamp", "Account", "Account.1", "Amount Paid", "Is Laundering"
    }
    missing = sorted(required - set(df_raw.columns))
    if missing:
        raise ValueError(f"原始資料缺少必要欄位：{missing}")
    if burn_in_days < 0:
        raise ValueError("burn_in_days 必須 >= 0")
    if not (0 < snapshot_ratio <= 1):
        raise ValueError("snapshot_ratio 必須位於 (0, 1]")
    if len(split_ratios) != 3 or not np.isclose(sum(split_ratios), 1.0):
        raise ValueError("split_ratios 必須有三個值且總和為 1")

    raw = df_raw.copy()
    raw.columns = raw.columns.str.strip()

    has_bank_cols = {"From Bank", "To Bank"}.issubset(raw.columns)
    if use_bank_account_id and has_bank_cols:
        name_orig = raw["From Bank"].astype(str) + "::" + raw["Account"].astype(str)
        name_dest = raw["To Bank"].astype(str) + "::" + raw["Account.1"].astype(str)
        node_id_mode = "bank::account"
    else:
        name_orig = raw["Account"].astype(str)
        name_dest = raw["Account.1"].astype(str)
        node_id_mode = "account"

    df = pd.DataFrame(index=np.arange(len(raw)))
    df["_tx_id"] = np.arange(len(raw), dtype=np.int64)
    df["nameOrig"] = name_orig.values
    df["nameDest"] = name_dest.values
    df["amount"] = pd.to_numeric(raw["Amount Paid"], errors="coerce").fillna(0.0).astype(float)
    if "Amount Received" in raw.columns:
        df["amount_received"] = pd.to_numeric(raw["Amount Received"], errors="coerce").fillna(0.0).astype(float)
    else:
        df["amount_received"] = df["amount"]
    df["isFraud"] = pd.to_numeric(raw["Is Laundering"], errors="coerce").fillna(0).astype(int)
    df["timestamp"] = pd.to_datetime(raw["Timestamp"], errors="coerce")
    if df["timestamp"].isna().any():
        raise ValueError("Timestamp 含有無法解析的值。")

    for raw_col, new_col in [
        ("From Bank", "from_bank"),
        ("To Bank", "to_bank"),
        ("Payment Currency", "payment_currency"),
        ("Receiving Currency", "receiving_currency"),
        ("Payment Format", "payment_format"),
    ]:
        df[new_col] = raw[raw_col].astype(str).values if raw_col in raw.columns else "UNKNOWN"

    df["same_bank"] = (df["from_bank"] == df["to_bank"]).astype(np.int8)
    df["same_currency"] = (df["payment_currency"] == df["receiving_currency"]).astype(np.int8)
    df["step"] = (df["timestamp"] - df["timestamp"].min()).dt.days.astype(int)
    df = df.sort_values(["step", "timestamp", "_tx_id"], kind="stable").reset_index(drop=True)

    all_nodes = pd.Index(
        pd.unique(pd.concat([df["nameOrig"], df["nameDest"]], ignore_index=True)),
        name="account",
    )
    node_to_idx = {node_id: idx for idx, node_id in enumerate(all_nodes)}
    node_table = pd.DataFrame({
        "node_idx": np.arange(len(all_nodes), dtype=np.int64),
        "node_id": all_nodes.astype(str),
    })

    burn_in_edges = df[df["step"] < burn_in_days].copy()
    cutoff_step = float(df["step"].max()) * snapshot_ratio
    snapshot = df[
        (df["step"] >= burn_in_days) & (df["step"] <= cutoff_step)
    ].copy()
    snapshot = snapshot.sort_values(["step", "timestamp", "_tx_id"], kind="stable").reset_index(drop=True)

    snapshot["edge_idx"] = np.arange(len(snapshot), dtype=np.int64)
    snapshot["src_idx"] = snapshot["nameOrig"].map(node_to_idx).astype(np.int64)
    snapshot["dst_idx"] = snapshot["nameDest"].map(node_to_idx).astype(np.int64)

    edge_index = torch.tensor(
        np.vstack([
            snapshot["src_idx"].to_numpy(),
            snapshot["dst_idx"].to_numpy(),
        ]),
        dtype=torch.long,
    )

    # Node-level label：snapshot 內任一 laundering transaction 的兩端皆標 1。
    y = np.zeros(len(all_nodes), dtype=np.int64)
    fraud_edges = snapshot[snapshot["isFraud"] == 1]
    fraud_nodes = pd.unique(pd.concat([
        fraud_edges["nameOrig"], fraud_edges["nameDest"]
    ], ignore_index=True))
    fraud_indices = [node_to_idx[n] for n in fraud_nodes if n in node_to_idx]
    y[fraud_indices] = 1

    # Active node 的首次出現時間直接由交易兩端建立，不依賴 NE ledger。
    event_orig = snapshot[["nameOrig", "timestamp"]].rename(columns={"nameOrig": "node_id"})
    event_dest = snapshot[["nameDest", "timestamp"]].rename(columns={"nameDest": "node_id"})
    first_seen = pd.concat([event_orig, event_dest], ignore_index=True).groupby("node_id")["timestamp"].min()

    if split_mode == "temporal_node":
        ordered_active_nodes = first_seen.sort_values(kind="stable").index.to_numpy()
    elif split_mode == "random":
        ordered_active_nodes = first_seen.index.to_numpy().copy()
        rng = np.random.default_rng(split_seed)
        rng.shuffle(ordered_active_nodes)
    else:
        raise ValueError("split_mode 只能是 'temporal_node' 或 'random'")

    active_indices = np.asarray([node_to_idx[n] for n in ordered_active_nodes], dtype=np.int64)
    n_active = len(active_indices)
    train_end = int(n_active * split_ratios[0])
    val_end = int(n_active * (split_ratios[0] + split_ratios[1]))

    train_mask = np.zeros(len(all_nodes), dtype=bool)
    val_mask = np.zeros(len(all_nodes), dtype=bool)
    test_mask = np.zeros(len(all_nodes), dtype=bool)
    train_mask[active_indices[:train_end]] = True
    val_mask[active_indices[train_end:val_end]] = True
    test_mask[active_indices[val_end:]] = True

    train_dirty_node_mask = train_mask & (y == 1)

    material = {
        "transactions": df,
        "burn_in_edges": burn_in_edges,
        "edge_table": snapshot,
        "node_table": node_table,
        "node_ids": all_nodes,
        "node_to_idx": node_to_idx,
        "edge_index": edge_index,
        "y": y,
        "train_mask": train_mask,
        "val_mask": val_mask,
        "test_mask": test_mask,
        "train_dirty_node_mask": train_dirty_node_mask,
        "num_nodes": len(all_nodes),
        "num_edges": len(snapshot),
        "config": {
            "node_id_mode": node_id_mode,
            "burn_in_days": burn_in_days,
            "snapshot_ratio": snapshot_ratio,
            "cutoff_step": cutoff_step,
            "split_mode": split_mode,
            "split_ratios": split_ratios,
            "split_seed": split_seed,
        },
    }

    print("✅ 胚料完成")
    print(f"node id mode : {node_id_mode}")
    print(f"nodes        : {material['num_nodes']:,}")
    print(f"snapshot edges: {material['num_edges']:,}")
    print(f"cutoff step  : {cutoff_step:.2f}")
    print(f"train/val/test active nodes: {train_mask.sum():,} / {val_mask.sum():,} / {test_mask.sum():,}")
    print(f"train-known dirty nodes: {train_dirty_node_mask.sum():,}")
    return material


## 3. 主函數二：NE `build_ne_features`

In [ ]:

import numpy as np
import pandas as pd


def _safe_log(x):
    arr = np.asarray(x, dtype=float)
    return np.log1p(np.abs(arr)) * np.sign(arr)


def _build_topography(material: dict) -> pd.DataFrame:
    all_nodes = material["node_ids"]
    burn = material["burn_in_edges"]
    topography = pd.DataFrame(index=all_nodes)
    topography.index.name = "account"
    topography["Z_0"] = 0.0
    inflow = burn.groupby("nameDest")["amount"].sum()
    outflow = burn.groupby("nameOrig")["amount"].sum()
    topography["Z_0"] = topography["Z_0"].add(inflow, fill_value=0).sub(outflow, fill_value=0)
    return topography


def _build_ledger(df_part: pd.DataFrame, topography: pd.DataFrame, window_size: int) -> pd.DataFrame:
    z0 = topography["Z_0"].to_dict()

    ledger_out = df_part[["_tx_id", "step", "timestamp", "nameOrig", "amount"]].copy()
    ledger_out.columns = ["_tx_id", "step", "timestamp", "account", "amount_change"]
    ledger_out["amount_change"] *= -1.0
    ledger_out["side"] = "orig"

    ledger_in = df_part[["_tx_id", "step", "timestamp", "nameDest", "amount"]].copy()
    ledger_in.columns = ["_tx_id", "step", "timestamp", "account", "amount_change"]
    ledger_in["side"] = "dest"

    ledger = pd.concat([ledger_out, ledger_in], ignore_index=True)
    ledger = ledger.sort_values(
        ["account", "timestamp", "step", "_tx_id", "side"], kind="stable"
    ).reset_index(drop=True)

    ledger["time_diff"] = ledger.groupby("account")["timestamp"].diff().dt.total_seconds().fillna(0.0)
    ledger["sigma_amount"] = ledger.groupby("account")["amount_change"].transform(
        lambda s: s.rolling(window=window_size, min_periods=1).std()
    ).fillna(0.0) + 1e-5
    ledger["sigma_time"] = ledger.groupby("account")["time_diff"].transform(
        lambda s: s.rolling(window=window_size, min_periods=1).std()
    ).fillna(0.0) + 1e-5

    ledger["Z_dynamic"] = ledger.groupby("account")["amount_change"].cumsum()
    ledger["Z_current"] = ledger["account"].map(z0).fillna(0.0) + ledger["Z_dynamic"]
    ledger["Z_prev"] = ledger.groupby("account")["Z_current"].shift(1)
    ledger["Z_prev"] = ledger["Z_prev"].fillna(ledger["account"].map(z0)).fillna(0.0)
    return ledger


def _attach_dynamic_heights(df_edges: pd.DataFrame, topography: pd.DataFrame, window_size: int):
    edges = df_edges.copy()
    ledger = _build_ledger(edges, topography, window_size)
    z_sender = ledger[ledger["side"] == "orig"][["_tx_id", "Z_prev"]].rename(columns={"Z_prev": "Z_sender"})
    z_receiver = ledger[ledger["side"] == "dest"][["_tx_id", "Z_prev"]].rename(columns={"Z_prev": "Z_receiver"})

    edges = edges.merge(z_sender, on="_tx_id", how="left", sort=False)
    edges = edges.merge(z_receiver, on="_tx_id", how="left", sort=False)
    z0 = topography["Z_0"].to_dict()
    edges["Z_sender"] = edges["Z_sender"].fillna(edges["nameOrig"].map(z0)).fillna(0.0)
    edges["Z_receiver"] = edges["Z_receiver"].fillna(edges["nameDest"].map(z0)).fillna(0.0)
    edges["F_spontaneous"] = edges["Z_sender"] - edges["Z_receiver"]
    edges = edges.sort_values("edge_idx", kind="stable").reset_index(drop=True)
    return edges, ledger


def _fk_physics(
    df_edges: pd.DataFrame,
    *,
    alpha: float,
    beta: float,
    impedance_decay: float,
) -> pd.Series:
    edge_id = df_edges["nameOrig"].astype(str) + "_" + df_edges["nameDest"].astype(str)
    freq = edge_id.groupby(edge_id).cumcount()
    area_a = np.log1p(freq) + 1.0

    degree_counts = df_edges["nameOrig"].value_counts().add(
        df_edges["nameDest"].value_counts(), fill_value=0
    )
    deg_orig = df_edges["nameOrig"].map(degree_counts).fillna(0.0)
    deg_dest = df_edges["nameDest"].map(degree_counts).fillna(0.0)
    iso_orig = 1.0 / np.log1p(deg_orig + 1.0)
    iso_dest = 1.0 / np.log1p(deg_dest + 1.0)
    roughness = (iso_orig * iso_dest) + 0.1
    k_static = roughness / area_a

    if "Delta_t" in df_edges.columns:
        k_dynamic = np.exp(-impedance_decay * df_edges["Delta_t"].clip(lower=0.01))
    else:
        k_dynamic = 0.0

    k_total = alpha * k_static + beta * k_dynamic
    return pd.Series(0.1 + np.log1p(k_total), index=df_edges.index, name="k_physics")


def _compute_physics_work(
    df_edges: pd.DataFrame,
    *,
    alpha: float,
    beta: float,
    impedance_decay: float,
) -> pd.DataFrame:
    edges = df_edges.sort_values(
        ["nameOrig", "timestamp", "step", "_tx_id"], kind="stable"
    ).copy()
    dt_days = edges.groupby("nameOrig")["timestamp"].diff().dt.total_seconds() / 86400.0
    edges["time_diff"] = dt_days.fillna(1.0).clip(lower=1e-4)
    edges["k_physics"] = _fk_physics(
        edges,
        alpha=alpha,
        beta=beta,
        impedance_decay=impedance_decay,
    )

    velocity = 1.0 / np.maximum(edges["time_diff"].to_numpy(), 1e-4)
    amount = edges["amount"].to_numpy()
    denominator = np.maximum(np.abs(edges["Z_sender"].to_numpy()), 1.0)
    grad_z = (edges["Z_receiver"].to_numpy() - edges["Z_sender"].to_numpy()) / denominator

    w_potential = amount * np.abs(grad_z)
    w_friction = edges["k_physics"].to_numpy() * amount * velocity
    w_kinetic = 0.5 * amount * velocity ** 2
    w_total = w_potential + w_friction + w_kinetic
    edges["Abnormal_Power"] = w_total * velocity
    edges["Abnormal_Pump_Work"] = edges["Abnormal_Power"]
    return edges.sort_values("edge_idx", kind="stable").reset_index(drop=True)


def _normalized_entropy_from_counts(df_in, group_col, value_col, out_name):
    eps = 1e-9
    tmp = (
        df_in[[group_col, value_col]].astype(str)
        .groupby([group_col, value_col]).size().rename("cnt").reset_index()
    )
    if tmp.empty:
        return pd.Series(dtype=float, name=out_name)
    tmp["total"] = tmp.groupby(group_col)["cnt"].transform("sum")
    tmp["k"] = tmp.groupby(group_col)["cnt"].transform("count")
    p = tmp["cnt"] / tmp["total"].clip(lower=1)
    tmp["entropy_part"] = -p * np.log(p + eps)
    ent = tmp.groupby(group_col)["entropy_part"].sum()
    k = tmp.groupby(group_col)["k"].max()
    result = ent / np.log(k.replace(1, np.e))
    result[k <= 1] = 0.0
    result = result.replace([np.inf, -np.inf], 0.0).fillna(0.0)
    result.name = out_name
    return result


def build_ne_features(
    material: dict,
    *,
    window_size: int = 5,
    alpha: float = 0.7922,
    beta: float = 0.7266,
    impedance_decay: float = 0.5,
) -> dict:
    """
    胚料 → Node + Edge（NE）特徵。

    這個函數完全不使用 laundering label。
    輸出的 node row 順序固定為 material['node_ids']；
    edge row 順序固定為 material['edge_table']['edge_idx']。
    """
    all_nodes = material["node_ids"]
    num_nodes = material["num_nodes"]
    edges = material["edge_table"].copy()

    topography = _build_topography(material)
    edges, ledger = _attach_dynamic_heights(edges, topography, window_size)
    edges = _compute_physics_work(
        edges,
        alpha=alpha,
        beta=beta,
        impedance_decay=impedance_decay,
    )

    # ----- Edge features -----
    edge_num = pd.DataFrame({
        "amount_paid_log": np.log1p(edges["amount"].to_numpy()),
        "amount_received_log": np.log1p(edges["amount_received"].to_numpy()),
        "slope_log": _safe_log(edges["Z_receiver"].to_numpy() - edges["Z_sender"].to_numpy()),
        "pump_work_log": _safe_log(edges["Abnormal_Pump_Work"].to_numpy()),
        "k_physics": edges["k_physics"].to_numpy(),
        "same_bank": edges["same_bank"].astype(float).to_numpy(),
        "same_currency": edges["same_currency"].astype(float).to_numpy(),
    }, index=edges["edge_idx"].to_numpy())

    cat_cols = ["payment_currency", "receiving_currency", "payment_format"]
    edge_cat = pd.get_dummies(edges[cat_cols].astype(str), prefix=cat_cols, dummy_na=False)
    edge_cat.index = edges["edge_idx"].to_numpy()
    edge_features = pd.concat([edge_num, edge_cat], axis=1).sort_index().astype(float)

    # ----- Physics node features -----
    pump_out_raw = edges.groupby("nameOrig")["Abnormal_Pump_Work"].agg(["max", "sum", "mean"]).fillna(0.0)
    pump_in_raw = edges.groupby("nameDest")["Abnormal_Pump_Work"].agg(["max", "sum", "mean"]).fillna(0.0)
    pump_out = pump_out_raw.apply(_safe_log)
    pump_in = pump_in_raw.apply(_safe_log)
    pump_out.columns = ["F_push_max", "F_push_sum", "F_push_mean"]
    pump_in.columns = ["F_pull_max", "F_pull_sum", "F_pull_mean"]

    period_change = edges.groupby("nameDest")["amount"].sum().add(
        edges.groupby("nameOrig")["amount"].sum() * -1.0,
        fill_value=0.0,
    )
    current_z_raw = topography["Z_0"].add(period_change, fill_value=0.0)
    current_z = pd.Series(_safe_log(current_z_raw), index=current_z_raw.index, name="Z_current")

    mean_sigmas = ledger.groupby("account")[["sigma_amount", "sigma_time"]].mean().fillna(1e-5)
    out_degree = np.log1p(np.bincount(edges["src_idx"].to_numpy(), minlength=num_nodes))
    in_degree = np.log1p(np.bincount(edges["dst_idx"].to_numpy(), minlength=num_nodes))
    out_degree = pd.Series(out_degree, index=all_nodes, name="out_degree_log")
    in_degree = pd.Series(in_degree, index=all_nodes, name="in_degree_log")

    node_physics = pd.concat(
        [current_z, pump_out, pump_in, mean_sigmas, out_degree, in_degree],
        axis=1,
    ).reindex(all_nodes).fillna(0.0)
    node_physics["sigma_amount"] = node_physics["sigma_amount"].replace(0.0, 1e-5)
    node_physics["sigma_time"] = node_physics["sigma_time"].replace(0.0, 1e-5)
    node_physics["Physics_Divergence"] = node_physics["F_push_sum"] - node_physics["F_pull_sum"]
    node_physics["Transit_Pressure"] = node_physics["F_push_sum"] + node_physics["F_pull_sum"]

    # ----- Dataset-native node context -----
    native_out = edges.groupby("nameOrig").agg(
        native_out_unique_to_bank=("to_bank", "nunique"),
        native_out_cross_bank_ratio=("same_bank", lambda x: 1.0 - float(np.mean(x))),
        native_out_unique_payment_currency=("payment_currency", "nunique"),
        native_out_cross_currency_ratio=("same_currency", lambda x: 1.0 - float(np.mean(x))),
        native_out_unique_payment_format=("payment_format", "nunique"),
    ).fillna(0.0)
    native_in = edges.groupby("nameDest").agg(
        native_in_unique_from_bank=("from_bank", "nunique"),
        native_in_cross_bank_ratio=("same_bank", lambda x: 1.0 - float(np.mean(x))),
        native_in_unique_receiving_currency=("receiving_currency", "nunique"),
        native_in_cross_currency_ratio=("same_currency", lambda x: 1.0 - float(np.mean(x))),
        native_in_unique_payment_format=("payment_format", "nunique"),
    ).fillna(0.0)

    for native_df in (native_out, native_in):
        for column in native_df.columns:
            if "unique" in column:
                native_df[column] = np.log1p(native_df[column].astype(float))
            elif "ratio" in column:
                native_df[column] = native_df[column].astype(float).clip(0.0, 1.0)

    native_out_format_entropy = _normalized_entropy_from_counts(
        edges, "nameOrig", "payment_format", "native_out_payment_format_entropy"
    )
    native_in_format_entropy = _normalized_entropy_from_counts(
        edges, "nameDest", "payment_format", "native_in_payment_format_entropy"
    )
    native_out_currency_entropy = _normalized_entropy_from_counts(
        edges, "nameOrig", "payment_currency", "native_out_payment_currency_entropy"
    )
    native_in_currency_entropy = _normalized_entropy_from_counts(
        edges, "nameDest", "receiving_currency", "native_in_receiving_currency_entropy"
    )

    ledger_time = ledger[["account", "timestamp", "step", "time_diff"]].copy()
    ledger_time["time_diff"] = ledger_time["time_diff"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    grouped_time = ledger_time.groupby("account")
    time_count = grouped_time.size().rename("native_event_count")
    active_days = grouped_time["step"].nunique().rename("native_active_days")
    first_time = grouped_time["timestamp"].min()
    last_time = grouped_time["timestamp"].max()
    span_days = ((last_time - first_time).dt.total_seconds() / 86400.0).clip(lower=0.0).rename("native_active_span_days")
    positive_gaps = ledger_time[ledger_time["time_diff"] > 0].groupby("account")["time_diff"].agg(["mean", "std", "min"]).fillna(0.0)
    positive_gaps.columns = ["native_mean_gap_sec", "native_std_gap_sec", "native_min_gap_sec"]

    native_time = pd.concat([time_count, active_days, span_days, positive_gaps], axis=1).fillna(0.0)
    native_time["native_event_count_log"] = np.log1p(native_time["native_event_count"])
    native_time["native_active_days_log"] = np.log1p(native_time["native_active_days"])
    native_time["native_active_span_days_log"] = np.log1p(native_time["native_active_span_days"])
    native_time["native_tx_per_active_day_log"] = np.log1p(
        native_time["native_event_count"] / native_time["native_active_days"].replace(0, 1)
    )
    native_time["native_mean_gap_log"] = np.log1p(native_time["native_mean_gap_sec"].clip(lower=0.0))
    native_time["native_min_gap_log"] = np.log1p(native_time["native_min_gap_sec"].clip(lower=0.0))
    burstiness = native_time["native_std_gap_sec"] / (native_time["native_mean_gap_sec"] + 1e-9)
    native_time["native_burstiness_log"] = np.log1p(
        burstiness.replace([np.inf, -np.inf], 0.0).fillna(0.0).clip(0.0, 100.0)
    )
    native_time = native_time[[
        "native_event_count_log",
        "native_active_days_log",
        "native_active_span_days_log",
        "native_tx_per_active_day_log",
        "native_mean_gap_log",
        "native_min_gap_log",
        "native_burstiness_log",
    ]]

    node_native = pd.concat([
        native_out,
        native_in,
        native_out_format_entropy,
        native_in_format_entropy,
        native_out_currency_entropy,
        native_in_currency_entropy,
        native_time,
    ], axis=1).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    node_native = node_native.reindex(all_nodes).fillna(0.0).astype(float)

    overlap = set(node_physics.columns) & set(node_native.columns)
    if overlap:
        node_native = node_native.rename(columns={c: f"native_ctx_{c}" for c in overlap})

    node_features = pd.concat([node_physics, node_native], axis=1)
    node_features = node_features.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)

    assert len(node_features) == material["num_nodes"]
    assert len(edge_features) == material["num_edges"]
    assert np.isfinite(node_features.to_numpy()).all()
    assert np.isfinite(edge_features.to_numpy()).all()

    print(" NE 完成")
    print(f"node physics dim: {node_physics.shape[1]}")
    print(f"node native dim : {node_native.shape[1]}")
    print(f"node NE dim     : {node_features.shape[1]}")
    print(f"edge NE dim     : {edge_features.shape[1]}")

    return {
        "node_features": node_features,
        "node_physics": node_physics,
        "node_native": node_native,
        "edge_features": edge_features,
        "processed_edges": edges,
        "topography": topography,
        "ledger": ledger,
        "params": {
            "window_size": window_size,
            "alpha": alpha,
            "beta": beta,
            "impedance_decay": impedance_decay,
        },
    }


## 4. Strict pack 與 V4.2 RW balanced 函數


In [ ]:

import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import RobustScaler
from torch_geometric.data import Data

PHYSICS_EDGE_COLS = ["slope_log", "pump_work_log", "k_physics"]

VALID_FEATURE_MODES = [
    "base",
    "base_ne",
    "base_rw_old",
    "base_nerw_old",
]


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False


def cleanup_gpu(label: str = ""):
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print(
            f"CUDA cleanup {label} | "
            f"allocated={torch.cuda.memory_allocated()/1024**3:.3f} GB | "
            f"reserved={torch.cuda.memory_reserved()/1024**3:.3f} GB"
        )


def _get_raw_edge_subset(ne: dict, material: dict) -> pd.DataFrame:
    """
    strict baseline edge = ne["edge_features"] 扣掉 physics edge columns。
    不重新生成 edge feature，只在 pack 層拆欄位。
    """
    expected_edge_index = np.arange(material["num_edges"])

    edge_df_full = ne["edge_features"].reindex(expected_edge_index).copy()
    edge_df_full = edge_df_full.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)

    missing = [c for c in PHYSICS_EDGE_COLS if c not in edge_df_full.columns]
    if missing:
        raise ValueError(f"ne['edge_features'] 找不到 physics edge columns: {missing}")

    return edge_df_full.drop(columns=PHYSICS_EDGE_COLS).copy()


def build_old_balanced_dirty_rw_features(
    material: dict,
    *,
    walkers_per_class: int = 20620,
    seed: int = 41,
    decay: float = 0.10,
    arrival_prob_mode: str = "edge_prob",
    count_start_node_visit: bool = False,
    stay_if_no_out: bool = True,
    feature_name: str = "dirty_exposure_score_oldrw_balanced",
) -> dict:
    """
    V4.2 oldRW / Balanced Dirty Exposure RW.

    特色：
        dirty walkers = walkers_per_class
        clean walkers = walkers_per_class
        total walkers = 2 * walkers_per_class

    僅使用 train positives / train negatives 作為 walker 起點，避免 val/test label leakage。

    輸出：
        score: DataFrame, shape = [num_nodes, 1]
    """
    rng = np.random.default_rng(seed)

    edge_table = material["edge_table"]
    num_nodes = int(material["num_nodes"])
    node_ids = pd.Index(material["node_ids"].astype(str))

    y = np.asarray(material["y"]).astype(int)
    train_mask = np.asarray(material["train_mask"]).astype(bool)

    dirty_pool = np.where(train_mask & (y == 1))[0]
    clean_pool = np.where(train_mask & (y == 0))[0]

    if len(dirty_pool) == 0:
        raise RuntimeError("train dirty pool 是空的，無法建立 dirty walkers。")
    if len(clean_pool) == 0:
        raise RuntimeError("train clean pool 是空的，無法建立 clean walkers。")

    print("\n🔧 建立 V4.2 oldRW day-level transition map...")

    required_cols = ["step", "src_idx", "dst_idx"]
    missing_cols = [c for c in required_cols if c not in edge_table.columns]
    if missing_cols:
        raise KeyError(f"material['edge_table'] 缺少欄位: {missing_cols}")

    tmp_edges = edge_table[required_cols].copy()
    tmp_edges["step"] = tmp_edges["step"].astype(int)

    edge_counts = (
        tmp_edges
        .groupby(["step", "src_idx", "dst_idx"])
        .size()
        .reset_index(name="cnt")
    )

    transition_by_step = {}
    steps = sorted(edge_counts["step"].unique().tolist())

    for step, g_step in edge_counts.groupby("step"):
        step_adj = {}

        for u, g_u in g_step.groupby("src_idx"):
            dsts = g_u["dst_idx"].to_numpy(dtype=np.int64)
            cnts = g_u["cnt"].to_numpy(dtype=np.float64)

            if len(dsts) == 0:
                continue

            probs = cnts / max(cnts.sum(), 1e-12)
            step_adj[int(u)] = (dsts, probs)

        transition_by_step[int(step)] = step_adj
        print(f"  step {int(step):02d}: active senders = {len(step_adj)}")

    print(f" transition map 完成，共 {len(transition_by_step)} 張照片")

    dirty_start = rng.choice(
        dirty_pool,
        size=walkers_per_class,
        replace=(len(dirty_pool) < walkers_per_class),
    )

    clean_start = rng.choice(
        clean_pool,
        size=walkers_per_class,
        replace=(len(clean_pool) < walkers_per_class),
    )

    walker_pos = np.concatenate([dirty_start, clean_start]).astype(np.int64)
    walker_is_dirty = np.concatenate([
        np.ones(walkers_per_class, dtype=bool),
        np.zeros(walkers_per_class, dtype=bool),
    ])

    known_dirty_mask = np.zeros(num_nodes, dtype=bool)
    known_dirty_mask[dirty_pool] = True

    num_walkers = len(walker_pos)
    retain = 1.0 - decay

    print(" oldRW walker setup")
    print(f"dirty seeds pool : {len(dirty_pool)}")
    print(f"clean seeds pool : {len(clean_pool)}")
    print(f"dirty walkers    : {walkers_per_class}")
    print(f"clean walkers    : {walkers_per_class}")
    print(f"total walkers    : {num_walkers}")
    print(f"decay / retain   : {decay:.4f} / {retain:.4f}")
    print(f"arrival mode     : {arrival_prob_mode}")

    dirty_exposure = np.zeros(num_nodes, dtype=np.float32)

    def add_dirty_visit(node_idx, p_update):
        dirty_exposure[node_idx] += float(p_update)

    if count_start_node_visit:
        for wi in range(num_walkers):
            if walker_is_dirty[wi]:
                add_dirty_visit(int(walker_pos[wi]), 1.0)

    print(" 開始 V4.2 oldRW balanced dirty exposure simulation...")

    for step_rank, step in enumerate(steps, start=1):
        step_adj = transition_by_step.get(int(step), {})

        dirty_exposure *= retain

        moved = 0
        became_dirty = 0
        dirty_moves = 0

        for wi in range(num_walkers):
            u = int(walker_pos[wi])

            if u not in step_adj:
                if not stay_if_no_out:
                    continue
                continue

            dsts, probs = step_adj[u]
            if len(dsts) == 0:
                continue

            choice = rng.choice(len(dsts), p=probs)
            v = int(dsts[choice])
            edge_p = float(probs[choice])

            walker_pos[wi] = v
            moved += 1

            # clean walker 走到 train-known dirty node 後變髒
            if (not walker_is_dirty[wi]) and known_dirty_mask[v]:
                walker_is_dirty[wi] = True
                became_dirty += 1

            if walker_is_dirty[wi]:
                dirty_moves += 1

                if arrival_prob_mode == "one":
                    p_update = 1.0
                elif arrival_prob_mode == "edge_prob":
                    p_update = edge_p
                else:
                    raise ValueError("arrival_prob_mode 必須是 'one' 或 'edge_prob'")

                add_dirty_visit(v, p_update)

        print(
            f"  step {int(step):02d} | moved={moved:5d}/{num_walkers} | "
            f"dirty_moves={dirty_moves:5d} | became_dirty={became_dirty:4d} | "
            f"dirty_now={walker_is_dirty.sum():5d}"
        )

    print(" V4.2 oldRW balanced dirty exposure simulation 完成")

    score_df = pd.DataFrame(
        {feature_name: dirty_exposure.astype(np.float32)},
        index=node_ids,
    )

    print(" oldRW feature dim:", score_df.shape[1])
    display(score_df.describe().T)

    return {
        "score": score_df,
        "dirty_exposure": dirty_exposure,
        "params": {
            "method": "V4.2_oldRW_balanced_dirty_exposure",
            "walkers_per_class": walkers_per_class,
            "total_walkers": int(2 * walkers_per_class),
            "dirty_walkers": int(walkers_per_class),
            "clean_walkers": int(walkers_per_class),
            "dirty_pool_size": int(len(dirty_pool)),
            "clean_pool_size": int(len(clean_pool)),
            "seed": int(seed),
            "decay": float(decay),
            "retain": float(retain),
            "arrival_prob_mode": arrival_prob_mode,
            "count_start_node_visit": bool(count_start_node_visit),
            "stay_if_no_out": bool(stay_if_no_out),
            "feature_name": feature_name,
        },
    }


def build_oldrw_feature_df(oldrw: dict, material: dict) -> pd.DataFrame:
    node_ids = pd.Index(material["node_ids"].astype(str))

    if "score" not in oldrw:
        raise KeyError("oldrw 裡找不到 score。")

    score_df = oldrw["score"].copy()

    if not isinstance(score_df, pd.DataFrame):
        arr = np.asarray(score_df).reshape(-1)
        if len(arr) != len(node_ids):
            raise ValueError(f"oldRW score 長度不一致: {len(arr)} vs {len(node_ids)}")
        score_df = pd.DataFrame({"dirty_exposure_score_oldrw_balanced": arr}, index=node_ids)

    if score_df.shape[1] != 1:
        raise ValueError(f"oldRW score 應該是 1 維，但目前是 {score_df.shape[1]} 維")

    aligned = score_df.reindex(node_ids)

    if aligned.isna().all().all() and len(score_df) == len(node_ids):
        aligned = pd.DataFrame(
            {score_df.columns[0]: score_df.iloc[:, 0].to_numpy()},
            index=node_ids,
        )

    aligned = aligned.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)

    return aligned


def _mode_uses_ne(mode: str) -> bool:
    return mode in ["base_ne", "base_nerw_old"]


def _mode_uses_oldrw(mode: str) -> bool:
    return mode in ["base_rw_old", "base_nerw_old"]


def pack_strict_ablation_features(
    material: dict,
    ne: dict,
    oldrw: dict | None = None,
    *,
    mode: str = "base",
    scale: bool = True,
    clip_value: float = 50.0,
    device=None,
) -> dict:
    """
    strict + oldRW 四組 mode:
        base:
            node = ne["node_native"]
            edge = raw edge subset

        base_ne:
            node = ne["node_features"]
            edge = ne["edge_features"]

        base_rw_old:
            node = ne["node_native"] + oldRW scalar dirty exposure
            edge = raw edge subset

        base_nerw_old:
            node = ne["node_features"] + oldRW scalar dirty exposure
            edge = ne["edge_features"]
    """
    if mode not in VALID_FEATURE_MODES:
        raise ValueError(f"mode 必須是 {VALID_FEATURE_MODES}")

    node_ids = pd.Index(material["node_ids"].astype(str))
    expected_edge_index = np.arange(material["num_edges"])

    if _mode_uses_ne(mode):
        node_df = ne["node_features"].reindex(node_ids).copy()
        edge_df = ne["edge_features"].reindex(expected_edge_index).copy()
    else:
        node_df = ne["node_native"].reindex(node_ids).copy()
        edge_df = _get_raw_edge_subset(ne, material)

    if _mode_uses_oldrw(mode):
        if oldrw is None:
            raise ValueError(f"{mode} 需要 oldrw，但 oldrw=None")

        oldrw_df = build_oldrw_feature_df(oldrw, material)

        overlap = set(node_df.columns) & set(oldrw_df.columns)
        if overlap:
            oldrw_df = oldrw_df.rename(columns={c: f"oldrw_{c}" for c in overlap})

        node_df = pd.concat([node_df, oldrw_df], axis=1)

    node_df = node_df.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)
    edge_df = edge_df.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)

    if len(node_df) != material["num_nodes"]:
        raise ValueError(f"Node feature row 數不一致: {len(node_df)} vs {material['num_nodes']}")

    if len(edge_df) != material["num_edges"]:
        raise ValueError(f"Edge feature row 數不一致: {len(edge_df)} vs {material['num_edges']}")

    if not np.isfinite(node_df.to_numpy()).all():
        raise ValueError("node features 含 NaN / inf")

    if not np.isfinite(edge_df.to_numpy()).all():
        raise ValueError("edge features 含 NaN / inf")

    raw_edge_dim = int(_get_raw_edge_subset(ne, material).shape[1])

    component_dims = {
        "node_dim": int(node_df.shape[1]),
        "edge_dim": int(edge_df.shape[1]),
        "node_native_dim": int(ne["node_native"].shape[1]),
        "node_physics_dim": int(ne["node_physics"].shape[1]) if _mode_uses_ne(mode) else 0,
        "rw_dim": 1 if _mode_uses_oldrw(mode) else 0,
        "rw_feature_mode": "oldrw_balanced_1d" if _mode_uses_oldrw(mode) else "none",
        "raw_edge_dim": raw_edge_dim,
        "physics_edge_dim": len(PHYSICS_EDGE_COLS) if _mode_uses_ne(mode) else 0,
    }

    node_array = node_df.to_numpy(dtype=np.float32)
    edge_array = edge_df.to_numpy(dtype=np.float32)
    train_mask = np.asarray(material["train_mask"]).astype(bool)

    node_scaler = None
    edge_scaler = None

    if scale:
        node_scaler = RobustScaler()
        node_scaler.fit(node_array[train_mask])
        node_array = node_scaler.transform(node_array)

        edge_scaler = RobustScaler()
        edge_array = edge_scaler.fit_transform(edge_array) if len(edge_array) else edge_array

    node_array = np.nan_to_num(node_array, nan=0.0, posinf=0.0, neginf=0.0)
    edge_array = np.nan_to_num(edge_array, nan=0.0, posinf=0.0, neginf=0.0)

    if clip_value is not None:
        node_array = np.clip(node_array, -clip_value, clip_value)
        edge_array = np.clip(edge_array, -clip_value, clip_value)

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device)

    data = Data(
        x=torch.tensor(node_array, dtype=torch.float32),
        edge_index=material["edge_index"].clone(),
        edge_attr=torch.tensor(edge_array, dtype=torch.float32),
        y=torch.tensor(material["y"], dtype=torch.float32),
        train_mask=torch.tensor(material["train_mask"], dtype=torch.bool),
        val_mask=torch.tensor(material["val_mask"], dtype=torch.bool),
        test_mask=torch.tensor(material["test_mask"], dtype=torch.bool),
    ).to(device)

    print(f"Pack 完成：{mode}")
    print(data)
    print(f"node dim = {data.x.size(1)}, edge dim = {data.edge_attr.size(1)}")
    print("component dims:", component_dims)

    return {
        "mode": mode,
        "data": data,
        "node_features_raw": node_df,
        "edge_features_raw": edge_df,
        "node_feature_names": list(node_df.columns),
        "edge_feature_names": list(edge_df.columns),
        "node_scaler": node_scaler,
        "edge_scaler": edge_scaler,
        "component_dims": component_dims,
        "dropped_physics_edge_cols": PHYSICS_EDGE_COLS if not _mode_uses_ne(mode) else [],
        "oldrw_params": oldrw.get("params", {}) if oldrw is not None else {},
        "device": device,
    }


def print_ablation_feature_contract(ne: dict, material: dict):
    raw_edge = _get_raw_edge_subset(ne, material)

    rows = [
        {
            "mode": "base",
            "node": "ne['node_native']",
            "node_dim": ne["node_native"].shape[1],
            "edge": "raw edge subset",
            "edge_dim": raw_edge.shape[1],
            "rw_dim": 0,
        },
        {
            "mode": "base_ne",
            "node": "ne['node_features']",
            "node_dim": ne["node_features"].shape[1],
            "edge": "ne['edge_features']",
            "edge_dim": ne["edge_features"].shape[1],
            "rw_dim": 0,
        },
        {
            "mode": "base_rw_old",
            "node": "ne['node_native'] + oldRW scalar dirty exposure",
            "node_dim": ne["node_native"].shape[1] + 1,
            "edge": "raw edge subset",
            "edge_dim": raw_edge.shape[1],
            "rw_dim": 1,
        },
        {
            "mode": "base_nerw_old",
            "node": "ne['node_features'] + oldRW scalar dirty exposure",
            "node_dim": ne["node_features"].shape[1] + 1,
            "edge": "ne['edge_features']",
            "edge_dim": ne["edge_features"].shape[1],
            "rw_dim": 1,
        },
    ]

    contract_df = pd.DataFrame(rows)
    display(contract_df)

    print("\nRaw edge columns:")
    for i, c in enumerate(raw_edge.columns):
        print(f"{i:02d}: {c}")

    print("\nPhysics edge columns:")
    print(PHYSICS_EDGE_COLS)

    return contract_df


## 5. GraphSAGE-Max / PhysicsSAGE-style 模型


In [ ]:
# ==========================================
# GraphSAGE-Max / PhysicsSAGE-style model and training
# ==========================================

import torch.nn.functional as F
from torch.nn import Sequential as Seq, Linear, ReLU, Dropout
from torch_geometric.nn import MessagePassing
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef,
)

SAGE_EPOCHS = 750
SAGE_HIDDEN_DIM = 64
SAGE_LR = 0.001
SAGE_WEIGHT_DECAY = 5e-4
EVAL_EVERY = 10


class PhysicsSAGEConv(MessagePassing):
    def __init__(self, node_in_dim, edge_in_dim, out_dim):
        super().__init__(aggr="max")
        self.msg_mlp = Seq(
            Linear(node_in_dim * 2 + edge_in_dim, out_dim),
            ReLU(),
            Linear(out_dim, out_dim),
        )
        self.update_mlp = Seq(
            Linear(node_in_dim + out_dim, out_dim),
            ReLU(),
        )

    def forward(self, x, edge_index, edge_attr):
        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr):
        return self.msg_mlp(torch.cat([x_i, x_j, edge_attr], dim=1))

    def update(self, aggr_out, x):
        return self.update_mlp(torch.cat([x, aggr_out], dim=1))


class PhysicsGNN(torch.nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=64):
        super().__init__()
        self.conv1 = PhysicsSAGEConv(node_dim, edge_dim, hidden_dim)
        self.conv2 = PhysicsSAGEConv(hidden_dim, edge_dim, hidden_dim)
        self.conv3 = PhysicsSAGEConv(hidden_dim, edge_dim, hidden_dim)
        self.classifier = Seq(
            Linear(hidden_dim, 32), ReLU(), Dropout(0.3), Linear(32, 1)
        )

    def forward(self, x, edge_index, edge_attr):
        h = F.relu(self.conv1(x, edge_index, edge_attr))
        h = F.relu(self.conv2(h, edge_index, edge_attr))
        h = F.relu(self.conv3(h, edge_index, edge_attr))
        return self.classifier(h).view(-1)


def _best_f1_threshold(labels, probs):
    precision, recall, thresholds = precision_recall_curve(labels, probs)

    if len(thresholds) == 0:
        return 0.5, 0.0, 0.0, 0.0

    f1 = 2 * precision[:-1] * recall[:-1] / (
        precision[:-1] + recall[:-1] + 1e-10
    )
    idx = int(np.argmax(f1))

    return (
        float(thresholds[idx]),
        float(f1[idx]),
        float(precision[idx]),
        float(recall[idx]),
    )


def _safe_roc_pr(labels, probs):
    if len(np.unique(labels)) <= 1:
        return 0.0, 0.0

    return (
        float(roc_auc_score(labels, probs)),
        float(average_precision_score(labels, probs)),
    )


@torch.no_grad()
def _eval_probs_labels(model, data, mask):
    model.eval()
    logits = model(data.x, data.edge_index, data.edge_attr)
    probs = torch.sigmoid(logits[mask]).detach().cpu().numpy()
    labels = data.y[mask].detach().cpu().numpy().astype(int)
    return probs, labels


def _threshold_metrics(labels, probs, threshold):
    pred = (probs >= threshold).astype(int)

    f1 = f1_score(labels, pred, zero_division=0)
    precision = precision_score(labels, pred, zero_division=0)
    recall = recall_score(labels, pred, zero_division=0)

    try:
        bal_acc = balanced_accuracy_score(labels, pred)
    except Exception:
        bal_acc = np.nan

    try:
        mcc = matthews_corrcoef(labels, pred)
    except Exception:
        mcc = np.nan

    tn, fp, fn, tp = confusion_matrix(labels, pred, labels=[0, 1]).ravel()

    return {
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "balanced_acc": float(bal_acc),
        "mcc": float(mcc),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }


def train_graphsage_one_mode(
    feature_pack: dict,
    material: dict,
    *,
    epochs: int = SAGE_EPOCHS,
    hidden_dim: int = SAGE_HIDDEN_DIM,
    lr: float = SAGE_LR,
    weight_decay: float = SAGE_WEIGHT_DECAY,
    eval_every: int = EVAL_EVERY,
    seed: int = 42,
    output_dir: str,
):
    set_seed(seed)

    mode = feature_pack["mode"]
    data = feature_pack["data"]

    model = PhysicsGNN(
        node_dim=data.x.size(1),
        edge_dim=data.edge_attr.size(1),
        hidden_dim=hidden_dim,
    ).to(data.x.device)

    num_pos = data.y[data.train_mask].sum().item()
    num_neg = data.train_mask.sum().item() - num_pos

    pos_weight = torch.tensor(
        [min(num_neg / (num_pos + 1e-5), 10.0)],
        dtype=torch.float32,
        device=data.x.device,
    )

    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    best_val_pr = -1.0
    best_state = None
    best_epoch = None
    history = []

    print("\n" + "=" * 90)
    print(f" Training GraphSAGE-Max mode={mode}")
    print("=" * 90)
    print(
        f"node_dim={data.x.size(1)}, edge_dim={data.edge_attr.size(1)}, "
        f"hidden_dim={hidden_dim}, pos_weight={pos_weight.item():.4f}"
    )

    logits = None
    loss = None

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        logits = model(data.x, data.edge_index, data.edge_attr)

        loss = criterion(
            logits[data.train_mask],
            data.y[data.train_mask].float(),
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if epoch % eval_every == 0:
            train_probs, train_labels = _eval_probs_labels(model, data, data.train_mask)
            val_probs, val_labels = _eval_probs_labels(model, data, data.val_mask)

            _, train_pr = _safe_roc_pr(train_labels, train_probs)
            val_roc, val_pr = _safe_roc_pr(val_labels, val_probs)

            marker = ""

            if val_pr > best_val_pr:
                best_val_pr = val_pr
                best_epoch = epoch
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }
                marker = " 🌟"

            history.append({
                "epoch": epoch,
                "loss": float(loss.item()),
                "train_pr_auc": float(train_pr),
                "val_roc_auc": float(val_roc),
                "val_pr_auc": float(val_pr),
            })

            print(
                f"Epoch {epoch:03d} | loss={loss.item():.4f} | "
                f"train PR={train_pr:.4f} | val ROC={val_roc:.4f} | val PR={val_pr:.4f}{marker}"
            )

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if best_state is not None:
        model.load_state_dict(best_state)

    val_probs, val_labels = _eval_probs_labels(model, data, data.val_mask)
    test_probs, test_labels = _eval_probs_labels(model, data, data.test_mask)

    val_roc, val_pr = _safe_roc_pr(val_labels, val_probs)
    test_roc, test_pr = _safe_roc_pr(test_labels, test_probs)

    threshold, val_f1, val_precision, val_recall = _best_f1_threshold(
        val_labels,
        val_probs,
    )

    test_metric = _threshold_metrics(test_labels, test_probs, threshold)

    print("\n" + "=" * 72)
    print(f"Best Val PR-AUC : {best_val_pr:.4f}")
    print(f"Best epoch      : {best_epoch}")
    print(f"Val ROC/PR      : {val_roc:.4f} / {val_pr:.4f}")
    print(f"Val threshold   : {threshold:.6f}")
    print(f"Val F1/P/R      : {val_f1:.4f} / {val_precision:.4f} / {val_recall:.4f}")
    print(f"Test ROC/PR     : {test_roc:.4f} / {test_pr:.4f}")
    print(
        f"Test F1/P/R     : "
        f"{test_metric['f1']:.4f} / {test_metric['precision']:.4f} / {test_metric['recall']:.4f}"
    )
    print(
        f"TP={test_metric['tp']}, FP={test_metric['fp']}, "
        f"FN={test_metric['fn']}, TN={test_metric['tn']}"
    )
    print("=" * 72)

    with torch.no_grad():
        model.eval()
        full_probs = torch.sigmoid(
            model(data.x, data.edge_index, data.edge_attr)
        ).detach().cpu().numpy()

    full_pred = (full_probs >= threshold).astype(int)

    scores = pd.DataFrame({
        "node_idx": np.arange(material["num_nodes"]),
        "account": material["node_ids"].astype(str),
        "y_true": material["y"],
        "score": full_probs,
        "pred": full_pred,
        "train_mask": material["train_mask"],
        "val_mask": material["val_mask"],
        "test_mask": material["test_mask"],
    })

    backbone = "GraphSAGE-Max"

    summary = pd.DataFrame([{
        "model": f"graphsage_max_{mode}",
        "backbone": backbone,
        "feature_mode": mode,
        "node_dim": int(data.x.size(1)),
        "edge_dim": int(data.edge_attr.size(1)),
        "seed": seed,
        "best_epoch": int(best_epoch) if best_epoch is not None else None,
        "best_val_pr_auc": float(best_val_pr),
        "val_roc_auc": float(val_roc),
        "val_pr_auc": float(val_pr),
        "val_best_threshold": float(threshold),
        "val_f1": float(val_f1),
        "val_precision": float(val_precision),
        "val_recall": float(val_recall),
        "test_roc_auc": float(test_roc),
        "test_pr_auc": float(test_pr),
        "test_f1_by_val_threshold": float(test_metric["f1"]),
        "test_precision_by_val_threshold": float(test_metric["precision"]),
        "test_recall_by_val_threshold": float(test_metric["recall"]),
        "test_balanced_acc": float(test_metric["balanced_acc"]),
        "test_mcc": float(test_metric["mcc"]),
        "test_tp": int(test_metric["tp"]),
        "test_fp": int(test_metric["fp"]),
        "test_fn": int(test_metric["fn"]),
        "test_tn": int(test_metric["tn"]),
        "hidden_dim": hidden_dim,
        "lr": lr,
        "weight_decay": weight_decay,
        "optimizer": "Adam",
        "best_checkpoint_metric": "Val PR-AUC",
        "threshold_policy": "Best-F1 threshold selected on Val, applied to Test",
    }])

    os.makedirs(output_dir, exist_ok=True)

    safe_backbone = "graphsage_max"
    summary_csv = os.path.join(output_dir, f"{safe_backbone}_{mode}_result_summary.csv")
    scores_csv = os.path.join(output_dir, f"{safe_backbone}_{mode}_scores_full.csv")
    ckpt_path = os.path.join(output_dir, f"checkpoint_{safe_backbone}_{mode}.pt")
    metrics_path = os.path.join(output_dir, f"metrics_{safe_backbone}_{mode}.json")
    history_path = os.path.join(output_dir, f"history_{safe_backbone}_{mode}.csv")

    summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
    scores.to_csv(scores_csv, index=False, encoding="utf-8-sig")
    pd.DataFrame(history).to_csv(history_path, index=False, encoding="utf-8-sig")

    model_state_cpu = {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }

    payload = {
        "model_state_dict": model_state_cpu,
        "mode": mode,
        "backbone": backbone,
        "summary": summary.to_dict(orient="records"),
        "history": history,
        "threshold": float(threshold),
        "best_val_metrics": {
            "epoch": int(best_epoch) if best_epoch is not None else None,
            "roc_auc": float(val_roc),
            "pr_auc": float(val_pr),
            "threshold": float(threshold),
            "f1": float(val_f1),
            "precision": float(val_precision),
            "recall": float(val_recall),
            "best_val_pr_auc": float(best_val_pr),
        },
        "test_metrics": {
            "roc_auc": float(test_roc),
            "pr_auc": float(test_pr),
            "threshold": float(threshold),
            "f1": float(test_metric["f1"]),
            "precision": float(test_metric["precision"]),
            "recall": float(test_metric["recall"]),
            "balanced_acc": float(test_metric["balanced_acc"]),
            "mcc": float(test_metric["mcc"]),
            "tp": int(test_metric["tp"]),
            "fp": int(test_metric["fp"]),
            "fn": int(test_metric["fn"]),
            "tn": int(test_metric["tn"]),
        },
        "component_dims": feature_pack["component_dims"],
        "node_feature_names": feature_pack["node_feature_names"],
        "edge_feature_names": feature_pack["edge_feature_names"],
        "dropped_physics_edge_cols": feature_pack["dropped_physics_edge_cols"],
        "config": {
            "epochs": epochs,
            "hidden_dim": hidden_dim,
            "lr": lr,
            "weight_decay": weight_decay,
            "seed": seed,
            "feature_mode": mode,
            "backbone": backbone,
            "optimizer": "Adam",
            "pos_weight_cap": 10.0,
            "best_checkpoint_metric": "Val PR-AUC",
            "threshold_policy": "Best-F1 threshold selected on Val, applied to Test",
            "grad_clip_norm": 1.0,
        },
        "paths": {
            "summary_csv": summary_csv,
            "scores_csv": scores_csv,
            "history_csv": history_path,
            "metrics_json": metrics_path,
            "checkpoint": ckpt_path,
        },
    }

    torch.save(payload, ckpt_path)

    metrics_payload = {
        "mode": mode,
        "backbone": backbone,
        "best_val_metrics": payload["best_val_metrics"],
        "test_metrics": payload["test_metrics"],
        "component_dims": feature_pack["component_dims"],
        "checkpoint_path": ckpt_path,
        "history_path": history_path,
        "summary_csv": summary_csv,
        "scores_csv": scores_csv,
        "config": payload["config"],
    }

    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics_payload, f, ensure_ascii=False, indent=2)

    print(f"\n Saved summary csv: {summary_csv}")
    print(f" Saved scores csv: {scores_csv}")
    print(f" Saved checkpoint: {ckpt_path}")
    print(f" Saved metrics json: {metrics_path}")
    print(f" Saved history csv: {history_path}")

    result = {
        "mode": mode,
        "checkpoint_path": ckpt_path,
        "metrics_path": metrics_path,
        "history_path": history_path,
        "summary_csv": summary_csv,
        "scores_csv": scores_csv,
        "summary": summary,
        "scores": scores,
        "threshold": threshold,
        "best_val_metrics": payload["best_val_metrics"],
        "test_metrics": payload["test_metrics"],
        "component_dims": dict(feature_pack["component_dims"]),
    }

    model.to("cpu")
    del model, optimizer, criterion, data, logits, loss, best_state
    cleanup_gpu(label=f"after train_graphsage_one_mode({mode})")

    return result


## 6. GINE-edge 模型


In [ ]:
# ==========================================
# GINE-edge model and training
# ==========================================

import torch.nn as nn
from torch_geometric.nn import GINEConv

GINE_EPOCHS = 750
GINE_HIDDEN_DIM = 64
GINE_LAYERS = 3
GINE_DROPOUT = 0.30
GINE_LR = 0.001
GINE_WEIGHT_DECAY = 5e-4


class GINEEdgeNodeModel(torch.nn.Module):
    def __init__(
        self,
        node_dim: int,
        edge_dim: int,
        hidden_dim: int = 64,
        num_layers: int = 3,
        dropout: float = 0.30,
    ):
        super().__init__()

        self.dropout = dropout
        self.node_proj = Linear(node_dim, hidden_dim)

        self.convs = torch.nn.ModuleList()
        self.norms = torch.nn.ModuleList()

        for _ in range(num_layers):
            mlp = Seq(
                Linear(hidden_dim, hidden_dim),
                ReLU(),
                Linear(hidden_dim, hidden_dim),
            )
            conv = GINEConv(
                nn=mlp,
                eps=0.0,
                train_eps=True,
                edge_dim=edge_dim,
            )
            self.convs.append(conv)
            self.norms.append(nn.LayerNorm(hidden_dim))

        self.classifier = Seq(
            Linear(hidden_dim, 32),
            ReLU(),
            Dropout(dropout),
            Linear(32, 1),
        )

    def forward(self, x, edge_index, edge_attr):
        h = self.node_proj(x)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index, edge_attr)
            h_new = norm(h_new)
            h_new = F.relu(h_new)
            h_new = F.dropout(h_new, p=self.dropout, training=self.training)
            h = h + h_new

        return self.classifier(h).view(-1)


def train_gine_edge_one_mode(
    feature_pack: dict,
    material: dict,
    *,
    epochs: int = GINE_EPOCHS,
    hidden_dim: int = GINE_HIDDEN_DIM,
    num_layers: int = GINE_LAYERS,
    dropout: float = GINE_DROPOUT,
    lr: float = GINE_LR,
    weight_decay: float = GINE_WEIGHT_DECAY,
    eval_every: int = EVAL_EVERY,
    seed: int = 42,
    output_dir: str,
):
    set_seed(seed)

    mode = feature_pack["mode"]
    data = feature_pack["data"]

    model = GINEEdgeNodeModel(
        node_dim=data.x.size(1),
        edge_dim=data.edge_attr.size(1),
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        dropout=dropout,
    ).to(data.x.device)

    num_pos = data.y[data.train_mask].sum().item()
    num_neg = data.train_mask.sum().item() - num_pos

    pos_weight = torch.tensor(
        [min(num_neg / (num_pos + 1e-5), 10.0)],
        dtype=torch.float32,
        device=data.x.device,
    )

    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    best_val_pr = -1.0
    best_state = None
    best_epoch = None
    history = []

    print("\n" + "=" * 90)
    print(f" Training GINE-edge mode={mode}")
    print("=" * 90)
    print(
        f"node_dim={data.x.size(1)}, edge_dim={data.edge_attr.size(1)}, "
        f"hidden_dim={hidden_dim}, layers={num_layers}, dropout={dropout}, "
        f"pos_weight={pos_weight.item():.4f}"
    )

    logits = None
    loss = None

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        logits = model(data.x, data.edge_index, data.edge_attr)

        loss = criterion(
            logits[data.train_mask],
            data.y[data.train_mask].float(),
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if epoch % eval_every == 0:
            train_probs, train_labels = _eval_probs_labels(model, data, data.train_mask)
            val_probs, val_labels = _eval_probs_labels(model, data, data.val_mask)

            _, train_pr = _safe_roc_pr(train_labels, train_probs)
            val_roc, val_pr = _safe_roc_pr(val_labels, val_probs)

            marker = ""

            if val_pr > best_val_pr:
                best_val_pr = val_pr
                best_epoch = epoch
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }
                marker = " 🌟"

            history.append({
                "epoch": epoch,
                "loss": float(loss.item()),
                "train_pr_auc": float(train_pr),
                "val_roc_auc": float(val_roc),
                "val_pr_auc": float(val_pr),
            })

            print(
                f"Epoch {epoch:03d} | loss={loss.item():.4f} | "
                f"train PR={train_pr:.4f} | val ROC={val_roc:.4f} | val PR={val_pr:.4f}{marker}"
            )

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if best_state is not None:
        model.load_state_dict(best_state)

    val_probs, val_labels = _eval_probs_labels(model, data, data.val_mask)
    test_probs, test_labels = _eval_probs_labels(model, data, data.test_mask)

    val_roc, val_pr = _safe_roc_pr(val_labels, val_probs)
    test_roc, test_pr = _safe_roc_pr(test_labels, test_probs)

    threshold, val_f1, val_precision, val_recall = _best_f1_threshold(
        val_labels,
        val_probs,
    )

    test_metric = _threshold_metrics(test_labels, test_probs, threshold)

    print("\n" + "=" * 72)
    print(f"Best Val PR-AUC : {best_val_pr:.4f}")
    print(f"Best epoch      : {best_epoch}")
    print(f"Val ROC/PR      : {val_roc:.4f} / {val_pr:.4f}")
    print(f"Val threshold   : {threshold:.6f}")
    print(f"Val F1/P/R      : {val_f1:.4f} / {val_precision:.4f} / {val_recall:.4f}")
    print(f"Test ROC/PR     : {test_roc:.4f} / {test_pr:.4f}")
    print(
        f"Test F1/P/R     : "
        f"{test_metric['f1']:.4f} / {test_metric['precision']:.4f} / {test_metric['recall']:.4f}"
    )
    print(
        f"TP={test_metric['tp']}, FP={test_metric['fp']}, "
        f"FN={test_metric['fn']}, TN={test_metric['tn']}"
    )
    print("=" * 72)

    with torch.no_grad():
        model.eval()
        full_probs = torch.sigmoid(
            model(data.x, data.edge_index, data.edge_attr)
        ).detach().cpu().numpy()

    full_pred = (full_probs >= threshold).astype(int)

    scores = pd.DataFrame({
        "node_idx": np.arange(material["num_nodes"]),
        "account": material["node_ids"].astype(str),
        "y_true": material["y"],
        "score": full_probs,
        "pred": full_pred,
        "train_mask": material["train_mask"],
        "val_mask": material["val_mask"],
        "test_mask": material["test_mask"],
    })

    backbone = "GINE-edge"

    summary = pd.DataFrame([{
        "model": f"gine_edge_{mode}",
        "backbone": backbone,
        "feature_mode": mode,
        "node_dim": int(data.x.size(1)),
        "edge_dim": int(data.edge_attr.size(1)),
        "seed": seed,
        "best_epoch": int(best_epoch) if best_epoch is not None else None,
        "best_val_pr_auc": float(best_val_pr),
        "val_roc_auc": float(val_roc),
        "val_pr_auc": float(val_pr),
        "val_best_threshold": float(threshold),
        "val_f1": float(val_f1),
        "val_precision": float(val_precision),
        "val_recall": float(val_recall),
        "test_roc_auc": float(test_roc),
        "test_pr_auc": float(test_pr),
        "test_f1_by_val_threshold": float(test_metric["f1"]),
        "test_precision_by_val_threshold": float(test_metric["precision"]),
        "test_recall_by_val_threshold": float(test_metric["recall"]),
        "test_balanced_acc": float(test_metric["balanced_acc"]),
        "test_mcc": float(test_metric["mcc"]),
        "test_tp": int(test_metric["tp"]),
        "test_fp": int(test_metric["fp"]),
        "test_fn": int(test_metric["fn"]),
        "test_tn": int(test_metric["tn"]),
        "hidden_dim": hidden_dim,
        "layers": num_layers,
        "dropout": dropout,
        "lr": lr,
        "weight_decay": weight_decay,
        "optimizer": "Adam",
        "best_checkpoint_metric": "Val PR-AUC",
        "threshold_policy": "Best-F1 threshold selected on Val, applied to Test",
    }])

    os.makedirs(output_dir, exist_ok=True)

    safe_backbone = "gine_edge"
    summary_csv = os.path.join(output_dir, f"{safe_backbone}_{mode}_result_summary.csv")
    scores_csv = os.path.join(output_dir, f"{safe_backbone}_{mode}_scores_full.csv")
    ckpt_path = os.path.join(output_dir, f"checkpoint_{safe_backbone}_{mode}.pt")
    metrics_path = os.path.join(output_dir, f"metrics_{safe_backbone}_{mode}.json")
    history_path = os.path.join(output_dir, f"history_{safe_backbone}_{mode}.csv")

    summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
    scores.to_csv(scores_csv, index=False, encoding="utf-8-sig")
    pd.DataFrame(history).to_csv(history_path, index=False, encoding="utf-8-sig")

    model_state_cpu = {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }

    payload = {
        "model_state_dict": model_state_cpu,
        "mode": mode,
        "backbone": backbone,
        "summary": summary.to_dict(orient="records"),
        "history": history,
        "threshold": float(threshold),
        "best_val_metrics": {
            "epoch": int(best_epoch) if best_epoch is not None else None,
            "roc_auc": float(val_roc),
            "pr_auc": float(val_pr),
            "threshold": float(threshold),
            "f1": float(val_f1),
            "precision": float(val_precision),
            "recall": float(val_recall),
            "best_val_pr_auc": float(best_val_pr),
        },
        "test_metrics": {
            "roc_auc": float(test_roc),
            "pr_auc": float(test_pr),
            "threshold": float(threshold),
            "f1": float(test_metric["f1"]),
            "precision": float(test_metric["precision"]),
            "recall": float(test_metric["recall"]),
            "balanced_acc": float(test_metric["balanced_acc"]),
            "mcc": float(test_metric["mcc"]),
            "tp": int(test_metric["tp"]),
            "fp": int(test_metric["fp"]),
            "fn": int(test_metric["fn"]),
            "tn": int(test_metric["tn"]),
        },
        "component_dims": feature_pack["component_dims"],
        "node_feature_names": feature_pack["node_feature_names"],
        "edge_feature_names": feature_pack["edge_feature_names"],
        "dropped_physics_edge_cols": feature_pack["dropped_physics_edge_cols"],
        "config": {
            "epochs": epochs,
            "hidden_dim": hidden_dim,
            "layers": num_layers,
            "dropout": dropout,
            "lr": lr,
            "weight_decay": weight_decay,
            "seed": seed,
            "feature_mode": mode,
            "backbone": backbone,
            "optimizer": "Adam",
            "pos_weight_cap": 10.0,
            "best_checkpoint_metric": "Val PR-AUC",
            "threshold_policy": "Best-F1 threshold selected on Val, applied to Test",
            "grad_clip_norm": 1.0,
        },
        "paths": {
            "summary_csv": summary_csv,
            "scores_csv": scores_csv,
            "history_csv": history_path,
            "metrics_json": metrics_path,
            "checkpoint": ckpt_path,
        },
    }

    torch.save(payload, ckpt_path)

    metrics_payload = {
        "mode": mode,
        "backbone": backbone,
        "best_val_metrics": payload["best_val_metrics"],
        "test_metrics": payload["test_metrics"],
        "component_dims": feature_pack["component_dims"],
        "checkpoint_path": ckpt_path,
        "history_path": history_path,
        "summary_csv": summary_csv,
        "scores_csv": scores_csv,
        "config": payload["config"],
    }

    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics_payload, f, ensure_ascii=False, indent=2)

    print(f"\n Saved summary csv: {summary_csv}")
    print(f" Saved scores csv: {scores_csv}")
    print(f" Saved checkpoint: {ckpt_path}")
    print(f" Saved metrics json: {metrics_path}")
    print(f" Saved history csv: {history_path}")

    result = {
        "mode": mode,
        "checkpoint_path": ckpt_path,
        "metrics_path": metrics_path,
        "history_path": history_path,
        "summary_csv": summary_csv,
        "scores_csv": scores_csv,
        "summary": summary,
        "scores": scores,
        "threshold": threshold,
        "best_val_metrics": payload["best_val_metrics"],
        "test_metrics": payload["test_metrics"],
        "component_dims": dict(feature_pack["component_dims"]),
    }

    model.to("cpu")
    del model, optimizer, criterion, data, logits, loss, best_state
    cleanup_gpu(label=f"after train_gine_edge_one_mode({mode})")

    return result


## 7. 建立 material / NE / oldRW

這裡只建一次 material、NE、oldRW，兩個 backbone 共用同一份 feature material。


In [ ]:
# ==========================================
# Build material / NE / V4.2 oldRW once
# ==========================================

SEED = 42

FEATURE_MODES_TO_RUN = [
    "base",
    "base_ne",
    "base_rw_old",
    "base_nerw_old",
]

OUTPUT_ROOT = "v43_strict_sage_gine_oldrw_balanced_outputs"
SAGE_OUTPUT_DIR = os.path.join(OUTPUT_ROOT, "graphsage_max")
GINE_OUTPUT_DIR = os.path.join(OUTPUT_ROOT, "gine_edge")

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(SAGE_OUTPUT_DIR, exist_ok=True)
os.makedirs(GINE_OUTPUT_DIR, exist_ok=True)

material = build_graph_material(
    df_raw,
    use_bank_account_id=True,
    burn_in_days=1,
    snapshot_ratio=0.8,
    split_mode="temporal_node",
    split_ratios=(0.6, 0.2, 0.2),
    split_seed=SEED,
)

ne = build_ne_features(
    material,
    window_size=5,
    alpha=0.7922,
    beta=0.7266,
    impedance_decay=0.5,
)

oldrw = build_old_balanced_dirty_rw_features(
    material,
    walkers_per_class=20620,
    seed=41,
    decay=0.10,
    arrival_prob_mode="edge_prob",
    count_start_node_visit=False,
    stay_if_no_out=True,
    feature_name="dirty_exposure_score_oldrw_balanced",
)

feature_contract_df = print_ablation_feature_contract(ne, material)
feature_contract_df.to_csv(
    os.path.join(OUTPUT_ROOT, "feature_contract_strict_sage_gine_oldrw.csv"),
    index=False,
    encoding="utf-8-sig",
)

with open(os.path.join(OUTPUT_ROOT, "oldrw_params.json"), "w", encoding="utf-8") as f:
    json.dump(oldrw["params"], f, ensure_ascii=False, indent=2)


## 8. 跑 GraphSAGE-Max 四組


In [ ]:
# ==========================================
# Train GraphSAGE-Max modes
# ==========================================

sage_run_records = []

for mode in FEATURE_MODES_TO_RUN:
    print("\n" + "#" * 80)
    print(f"Running GraphSAGE-Max strict oldRW mode: {mode}")
    print("#" * 80)

    feature_pack = pack_strict_ablation_features(
        material,
        ne,
        oldrw=oldrw,
        mode=mode,
        scale=True,
        clip_value=50.0,
    )

    result = train_graphsage_one_mode(
        feature_pack,
        material,
        epochs=SAGE_EPOCHS,
        hidden_dim=SAGE_HIDDEN_DIM,
        lr=SAGE_LR,
        weight_decay=SAGE_WEIGHT_DECAY,
        eval_every=EVAL_EVERY,
        seed=SEED,
        output_dir=SAGE_OUTPUT_DIR,
    )

    sage_run_records.append({
        "backbone": "GraphSAGE-Max",
        "mode": mode,
        "checkpoint_path": result["checkpoint_path"],
        "metrics_path": result["metrics_path"],
        "history_path": result["history_path"],
        "summary_csv": result["summary_csv"],
        "scores_csv": result["scores_csv"],
        **result["component_dims"],
    })

    del feature_pack
    del result

    cleanup_gpu(label=f"after GraphSAGE mode={mode}")

sage_run_records_df = pd.DataFrame(sage_run_records)
sage_run_records_path = os.path.join(SAGE_OUTPUT_DIR, "run_records_graphsage_max_strict_oldrw.csv")
sage_run_records_df.to_csv(sage_run_records_path, index=False, encoding="utf-8-sig")

display(sage_run_records_df)
print("Saved:", sage_run_records_path)


## 9. 跑 GINE-edge 四組


In [ ]:
# ==========================================
# Train GINE-edge modes
# ==========================================

gine_run_records = []

for mode in FEATURE_MODES_TO_RUN:
    print("\n" + "#" * 80)
    print(f"Running GINE-edge strict oldRW mode: {mode}")
    print("#" * 80)

    feature_pack = pack_strict_ablation_features(
        material,
        ne,
        oldrw=oldrw,
        mode=mode,
        scale=True,
        clip_value=50.0,
    )

    result = train_gine_edge_one_mode(
        feature_pack,
        material,
        epochs=GINE_EPOCHS,
        hidden_dim=GINE_HIDDEN_DIM,
        num_layers=GINE_LAYERS,
        dropout=GINE_DROPOUT,
        lr=GINE_LR,
        weight_decay=GINE_WEIGHT_DECAY,
        eval_every=EVAL_EVERY,
        seed=SEED,
        output_dir=GINE_OUTPUT_DIR,
    )

    gine_run_records.append({
        "backbone": "GINE-edge",
        "mode": mode,
        "checkpoint_path": result["checkpoint_path"],
        "metrics_path": result["metrics_path"],
        "history_path": result["history_path"],
        "summary_csv": result["summary_csv"],
        "scores_csv": result["scores_csv"],
        **result["component_dims"],
    })

    del feature_pack
    del result

    cleanup_gpu(label=f"after GINE mode={mode}")

gine_run_records_df = pd.DataFrame(gine_run_records)
gine_run_records_path = os.path.join(GINE_OUTPUT_DIR, "run_records_gine_edge_strict_oldrw.csv")
gine_run_records_df.to_csv(gine_run_records_path, index=False, encoding="utf-8-sig")

display(gine_run_records_df)
print("Saved:", gine_run_records_path)


## 10. 最終綜合表：GraphSAGE + GINE

讀兩個 output directory 的 metrics json，整理同一張表。


In [ ]:
# ==========================================
# Final combined summary: GraphSAGE + GINE strict oldRW
# ==========================================

EXPECTED_MODES = ["base", "base_ne", "base_rw_old", "base_nerw_old"]

BACKBONE_SPECS = [
    {
        "backbone": "GraphSAGE-Max",
        "dir": SAGE_OUTPUT_DIR,
        "prefix": "graphsage_max",
    },
    {
        "backbone": "GINE-edge",
        "dir": GINE_OUTPUT_DIR,
        "prefix": "gine_edge",
    },
]

def _safe_get(d, key, default=np.nan):
    if isinstance(d, dict):
        return d.get(key, default)
    return default

rows = []

for spec in BACKBONE_SPECS:
    for mode in EXPECTED_MODES:
        metrics_path = os.path.join(spec["dir"], f"metrics_{spec['prefix']}_{mode}.json")

        if not os.path.exists(metrics_path):
            print(f"⚠️ 找不到 {metrics_path}，略過 {spec['backbone']} / {mode}")
            continue

        with open(metrics_path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        val = obj.get("best_val_metrics", {})
        test = obj.get("test_metrics", {})
        dims = obj.get("component_dims", {})
        config = obj.get("config", {})

        rows.append({
            "Backbone": spec["backbone"],
            "Feature Mode": mode,
            "Node Dim": _safe_get(dims, "node_dim"),
            "Edge Dim": _safe_get(dims, "edge_dim"),
            "RW Dim": _safe_get(dims, "rw_dim"),
            "RW Mode": _safe_get(dims, "rw_feature_mode"),
            "Best Val Epoch": _safe_get(val, "epoch"),

            "Val F1": _safe_get(val, "f1"),
            "Val Recall": _safe_get(val, "recall"),
            "Val Precision": _safe_get(val, "precision"),
            "Val PR-AUC": _safe_get(val, "pr_auc"),
            "Val ROC-AUC": _safe_get(val, "roc_auc"),

            "Test F1": _safe_get(test, "f1"),
            "Test Recall": _safe_get(test, "recall"),
            "Test Precision": _safe_get(test, "precision"),
            "Test PR-AUC": _safe_get(test, "pr_auc"),
            "Test ROC-AUC": _safe_get(test, "roc_auc"),
            "Test MCC": _safe_get(test, "mcc"),
            "Test Threshold": _safe_get(test, "threshold"),
            "TP": _safe_get(test, "tp"),
            "FP": _safe_get(test, "fp"),
            "FN": _safe_get(test, "fn"),
            "TN": _safe_get(test, "tn"),

            "Hidden Dim": _safe_get(config, "hidden_dim"),
            "Dropout": _safe_get(config, "dropout"),
            "Optimizer": _safe_get(config, "optimizer"),
            "Best Checkpoint Metric": _safe_get(config, "best_checkpoint_metric"),
        })

if len(rows) == 0:
    raise RuntimeError("沒有讀到任何 metrics json，請先跑訓練 cell。")

paper_table = pd.DataFrame(rows)
paper_table["Feature Mode"] = pd.Categorical(
    paper_table["Feature Mode"],
    categories=EXPECTED_MODES,
    ordered=True,
)

paper_table = paper_table.sort_values(["Backbone", "Feature Mode"]).reset_index(drop=True)

# 對每個 backbone 分別計算 vs base / vs NE
for backbone in paper_table["Backbone"].unique():
    sub_idx = paper_table["Backbone"] == backbone
    sub = paper_table[sub_idx]

    if "base" in sub["Feature Mode"].astype(str).values:
        base_row = sub[sub["Feature Mode"].astype(str) == "base"].iloc[0]

        for col in ["Test F1", "Test Recall", "Test Precision", "Test PR-AUC"]:
            paper_table.loc[sub_idx, f"Δ {col} vs Base"] = (
                paper_table.loc[sub_idx, col] - base_row[col]
            )

    if "base_ne" in sub["Feature Mode"].astype(str).values:
        ne_row = sub[sub["Feature Mode"].astype(str) == "base_ne"].iloc[0]

        for col in ["Test F1", "Test Recall", "Test Precision", "Test PR-AUC"]:
            paper_table.loc[sub_idx, f"Δ {col} vs NE"] = (
                paper_table.loc[sub_idx, col] - ne_row[col]
            )

numeric_cols = paper_table.select_dtypes(include=[np.number]).columns
paper_table[numeric_cols] = paper_table[numeric_cols].round(6)

combined_path = os.path.join(
    OUTPUT_ROOT,
    "paper_table_sage_gine_strict_oldrw_test_f1_recall_precision_prauc.csv",
)

paper_table.to_csv(combined_path, index=False, encoding="utf-8-sig")

print("=== GraphSAGE + GINE strict oldRW Paper Table ===")
display(paper_table)

print("\nSaved:")
print(" -", combined_path)


## 11. 顯示每組特徵欄位

如果結果怪怪的，先看這裡確認 pack 沒有偷改。


In [ ]:
# ==========================================
# Inspect feature names from saved checkpoints
# ==========================================

for spec in BACKBONE_SPECS:
    print("\n" + "#" * 100)
    print(spec["backbone"])
    print("#" * 100)

    for mode in EXPECTED_MODES:
        ckpt_path = os.path.join(spec["dir"], f"checkpoint_{spec['prefix']}_{mode}.pt")

        if not os.path.exists(ckpt_path):
            print(f"找不到 {ckpt_path}")
            continue

        ckpt = torch.load(ckpt_path, map_location="cpu")

        print("\n" + "=" * 80)
        print(f"Mode: {mode}")
        print("component_dims:", ckpt["component_dims"])

        print("\nNode features:")
        for i, c in enumerate(ckpt["node_feature_names"]):
            print(f"{i:02d}: {c}")

        print("\nEdge features:")
        for i, c in enumerate(ckpt["edge_feature_names"]):
            print(f"{i:02d}: {c}")

        print("Dropped physics edge cols:", ckpt["dropped_physics_edge_cols"])
